# StringSense ABSA Labeling Notebook

This notebook shows how to process the latest review dataset and generate:

1. `nlp_absa_long_dataset_latest.csv`
2. `nlp_absa_high_confidence_latest.csv`

The notebook focuses on:

- loading the latest archive
- normalization
- clause splitting
- domain dictionary loading
- rule-based aspect detection
- mention labeling
- sentiment labeling
- exporting the two training datasets

All markdown and code comments are written in English.


## 1. Install Required Libraries

Run this cell first if your environment does not already have the required packages.


In [ ]:
# Install the required libraries if needed
%pip install pandas numpy jieba openpyxl


## 2. Import Libraries and Define File Paths

In [ ]:
import json
import zipfile
import hashlib
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import jieba

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data" / "archive_latest.zip").exists():
    workspace_candidate = BASE_DIR / "ml" / "nlp-workbench-latest"
    if (workspace_candidate / "data" / "archive_latest.zip").exists():
        BASE_DIR = workspace_candidate

DATA_DIR = BASE_DIR / "data"
ARCHIVE_ZIP = DATA_DIR / "archive_latest.zip"
DICT_CSV = DATA_DIR / "domain_dictionary_optimized_v8.csv"
NORM_CSV = DATA_DIR / "normalization_rules_v8.csv"

OUTPUT_DIR = DATA_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LONG_OUT = OUTPUT_DIR / "nlp_absa_long_dataset_latest.csv"
HIGH_OUT = OUTPUT_DIR / "nlp_absa_high_confidence_latest.csv"


## 3. Load the Latest Archive

The archive contains the latest raw badminton string reviews.


In [ ]:
with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
    with z.open("badminton_strings_data.json") as f:
        raw_data = json.load(f)

strings = raw_data["strings"]
print("Number of strings:", len(strings))


## 4. Load the Optimized Domain Dictionary and Normalization Rules

In [ ]:
dict_df = pd.read_csv(DICT_CSV)
norm_df = pd.read_csv(NORM_CSV)

print("Dictionary rows:", len(dict_df))
print("Normalization rule rows:", len(norm_df))

dict_df.head()


## 5. Register Custom Terms into jieba

This helps keep badminton-specific terms intact during tokenization.


In [ ]:
custom_terms = (
    dict_df["term"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

for term in custom_terms:
    if term:
        jieba.add_word(term)

print("Custom terms registered:", len(custom_terms))


## 6. Define Text Normalization and Clause Splitting

Normalization reduces surface variation.
Clause splitting is used because one review may express multiple aspects.


In [ ]:
normalization_rules = list(
    zip(
        norm_df["pattern"].astype(str).tolist(),
        norm_df["replacement"].astype(str).tolist()
    )
)

def normalize_text(text: str) -> str:
    text = str(text)
    for pattern, replacement in normalization_rules:
        text = re.sub(pattern, replacement, text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def split_into_clauses(text: str):
    text = normalize_text(text)
    parts = re.split(r"[。！？；;.!?\n]+|但是|但|不过|然而|就是|而且|同时", text)
    parts = [p.strip(" ，,：:、 ") for p in parts if p and p.strip(" ，,：:、 ")]
    return parts


## 7. Build the Aspect Lexicon

The dictionary is converted into three sets for each aspect:

- aspect terms
- positive terms
- negative terms


In [ ]:
aspect_lexicon = defaultdict(lambda: {
    "aspect_terms": set(),
    "positive_terms": set(),
    "negative_terms": set()
})

for _, row in dict_df.iterrows():
    aspect = str(row["aspect"]).strip()
    term_type = str(row.get("term_type", "")).strip()
    term = str(row["term"]).strip()
    polarity = str(row.get("polarity", "")).strip().lower()

    if not aspect or not term or term == "nan":
        continue

    if term_type == "aspect_term":
        aspect_lexicon[aspect]["aspect_terms"].add(term)

    if polarity == "positive":
        aspect_lexicon[aspect]["positive_terms"].add(term)
    elif polarity == "negative":
        aspect_lexicon[aspect]["negative_terms"].add(term)

aspect_list = sorted(aspect_lexicon.keys())
aspect_list


## 8. Define Helper Functions for Metadata Extraction

In [ ]:
def has_tension_mention(text: str):
    return "yes" if re.search(r"\b\d{1,2}\s*(?:lbs?|LB|磅)\b|[0-9]{1,2}\s*磅", text, flags=re.IGNORECASE) else "no"

def extract_tension(text: str):
    m = re.search(r"([1-4]?\d(?:\.\d)?)\s*(?:lbs?|LB|磅)", text, flags=re.IGNORECASE)
    if m:
        try:
            return float(m.group(1))
        except Exception:
            return np.nan
    return np.nan

def has_price_mention(text: str):
    return "yes" if re.search(r"\bRM\s*\d+|\d+\s*rm|价格|价位|贵|便宜|小贵|偏贵|不值|值这个价", text, flags=re.IGNORECASE) else "no"

def deterministic_split(key: str):
    h = int(hashlib.md5(key.encode("utf-8")).hexdigest(), 16) % 100
    if h < 80:
        return "train"
    elif h < 90:
        return "val"
    return "test"


## 9. Define the Core Rule-Based Labeling Function

This function produces:

- `mention_flag`
- `sentiment_id`
- `label_text`
- `label_id`
- `needs_manual_review`

### Label meaning

- `not_mentioned`: aspect not found
- `mentioned`: aspect found but polarity unclear
- `positive`: aspect found and positive
- `negative`: aspect found and negative
- `mixed`: both positive and negative evidence found


In [ ]:
def classify_review_aspect(review_text: str, clauses, aspect: str):
    lex = aspect_lexicon[aspect]

    pos_hits_total = 0
    neg_hits_total = 0
    matched_clauses = 0

    for clause in clauses:
        aspect_hits = sum(1 for t in lex["aspect_terms"] if t in clause)
        pos_hits = sum(1 for t in lex["positive_terms"] if t in clause)
        neg_hits = sum(1 for t in lex["negative_terms"] if t in clause)

        if aspect_hits or pos_hits or neg_hits:
            matched_clauses += 1
            pos_hits_total += pos_hits
            neg_hits_total += neg_hits

    if matched_clauses == 0:
        return {
            "label_text": "not_mentioned",
            "label_id": 0,
            "mention_flag": 0,
            "sentiment_id": np.nan,
            "needs_manual_review": 0
        }

    if pos_hits_total == 0 and neg_hits_total == 0:
        return {
            "label_text": "mentioned",
            "label_id": 1,
            "mention_flag": 1,
            "sentiment_id": np.nan,
            "needs_manual_review": 1
        }

    if pos_hits_total > 0 and neg_hits_total == 0:
        return {
            "label_text": "positive",
            "label_id": 2,
            "mention_flag": 1,
            "sentiment_id": 1.0,
            "needs_manual_review": 0
        }

    if neg_hits_total > 0 and pos_hits_total == 0:
        return {
            "label_text": "negative",
            "label_id": 3,
            "mention_flag": 1,
            "sentiment_id": -1.0,
            "needs_manual_review": 0
        }

    return {
        "label_text": "mixed",
        "label_id": 4,
        "mention_flag": 1,
        "sentiment_id": 0.0,
        "needs_manual_review": 1
    }


## 10. Generate the Long Dataset

This dataset includes all review-aspect pairs and is mainly used for mention detection.


In [ ]:
rows = []

for idx, item in enumerate(strings, start=1):
    string_name = item.get("name", "")
    string_id = f"S{idx:03d}"

    for r in item.get("reviews", []):
        review_text = (r.get("content") or "").strip()
        if not review_text:
            continue

        review_id_raw = r.get("review_id", "")
        review_id = f"R{review_id_raw}" if str(review_id_raw) else f"RUNK_{idx}"

        rating_label = r.get("rating_label", "")
        likes_count = r.get("likes", 0) or 0
        review_date = r.get("review_date", "")
        source_url = r.get("source_url", "")

        normalized_text = normalize_text(review_text)
        clauses = split_into_clauses(normalized_text)

        tension_flag = has_tension_mention(normalized_text)
        price_flag = has_price_mention(normalized_text)
        extracted_tension = extract_tension(normalized_text)

        for aspect in aspect_list:
            cls = classify_review_aspect(normalized_text, clauses, aspect)
            sample_id = f"{review_id}_{aspect}"
            split = deterministic_split(sample_id)

            rows.append({
                "sample_id": sample_id,
                "split": split,
                "review_id": review_id,
                "string_id": string_id,
                "string_name": string_name,
                "aspect": aspect,
                "label_text": cls["label_text"],
                "label_id": cls["label_id"],
                "mention_flag": cls["mention_flag"],
                "sentiment_id": cls["sentiment_id"],
                "needs_manual_review": cls["needs_manual_review"],
                "review_text": normalized_text,
                "rating_label": rating_label,
                "has_tension_mention": tension_flag,
                "has_price_mention": price_flag,
                "extracted_tension": extracted_tension,
                "likes_count": likes_count,
                "review_date": review_date,
                "source_url": source_url
            })

long_df = pd.DataFrame(rows)
print("Long dataset rows:", len(long_df))
long_df.head()


## 11. Generate the High-Confidence Dataset

This dataset keeps only clean rows for model-based sentiment training.

Kept labels:
- `not_mentioned`
- `positive`
- `negative`

Removed:
- `mentioned`
- `mixed`
- rows marked for manual review


In [ ]:
high_df = long_df[
    (long_df["needs_manual_review"] == 0) &
    (long_df["label_text"].isin(["not_mentioned", "positive", "negative"]))
].copy()

print("High-confidence dataset rows:", len(high_df))
high_df.head()


## 12. Inspect Label Distribution

In [ ]:
print("Long dataset label distribution:")
print(long_df["label_text"].value_counts())

print("\nHigh-confidence label distribution:")
print(high_df["label_text"].value_counts())


## 13. Save the Two Latest Files

In [ ]:
long_df.to_csv(LONG_OUT, index=False, encoding="utf-8-sig")
high_df.to_csv(HIGH_OUT, index=False, encoding="utf-8-sig")

print("Saved:")
print(LONG_OUT)
print(HIGH_OUT)


## 14. Example: Inspect One Review Across All Aspects

This is useful to understand how mention and sentiment labels are produced.


In [ ]:
example_review_id = long_df.iloc[0]["review_id"]
example_df = long_df[long_df["review_id"] == example_review_id].copy()
example_df[[
    "review_id", "string_name", "aspect", "label_text", "mention_flag", "sentiment_id", "needs_manual_review", "review_text"
]].head(20)


## 15. Final Summary

Use the outputs as follows:

- `nlp_absa_long_dataset_latest.csv` for mention model training
- `nlp_absa_high_confidence_latest.csv` for sentiment model training

This notebook is intended to show how the labels are generated, not only to export the final files.
